<a href="https://colab.research.google.com/github/Sanjidh090/Trends_analysis_tool/blob/main/Avadakedavra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install trendspy pandas

In [ ]:
"""
MARKET DEMAND INTELLIGENCE ENGINE — Service Focused
Find WHERE people need your services (not viral trends)

Install:
    pip install trendspy pandas

Run:
    python market_demand_engine.py
"""

import time
import datetime
import json
import pandas as pd
from trendspy import Trends

# ── CONFIG ───────────────────────────────────────────────
GEO = "US"          # Change to target market (BD, US, AE, SG, etc.)
TIMEFRAME = "today 3-d"
DELAY = 2.0
OUTPUT_FILE = "market_demand_report.json"

# 🔥 Your actual services (EDIT THIS)
SERVICE_KEYWORDS = [
    "web development service",
    "seo service",
    "ai automation",
    "data analysis service",
    "digital marketing agency",
    "app development company",
    "pcb design service",
]

# ── UTIL ────────────────────────────────────────────────
def sleep():
    time.sleep(DELAY)

def log(msg):
    print(msg)

# ── DATA FETCHING ───────────────────────────────────────
def fetch_regions(tr, keyword):
    sleep()
    try:
        df = tr.interest_by_region([keyword], geo=GEO, resolution="REGION")
        if df is None or df.empty:
            return []
        df = df.sort_values(keyword, ascending=False).head(8)
        return [{"region": row["geoName"], "value": int(row[keyword])}
                for _, row in df.iterrows()]
    except:
        return []

def fetch_related(tr, keyword):
    sleep()
    try:
        r = tr.related_queries([keyword], geo=GEO)
        top_df = r.get(keyword, {}).get("top", pd.DataFrame())
        rising_df = r.get(keyword, {}).get("rising", pd.DataFrame())

        return {
            "top": top_df["query"].tolist()[:5] if not top_df.empty else [],
            "rising": rising_df["query"].tolist()[:5] if not rising_df.empty else [],
        }
    except:
        return {"top": [], "rising": []}

def fetch_iot(tr, keyword):
    sleep()
    try:
        df = tr.interest_over_time([keyword], geo=GEO, timeframe=TIMEFRAME)
        if df is None or df.empty:
            return {}

        s = df[keyword].tolist()
        return {
            "trend": s,
            "peak": max(s),
            "avg": round(sum(s)/len(s), 1),
            "latest": s[-1],
        }
    except:
        return {}

# ── OPPORTUNITY SCORING ─────────────────────────────────
def score_opportunity(regions, rising, iot):
    score = 0

    # Demand strength (40)
    score += min(40, sum(r["value"] for r in regions) / 10)

    # Rising queries (30)
    score += min(30, len(rising) * 6)

    # Trend momentum (30)
    if iot:
        if iot["latest"] > iot["avg"]:
            score += 20
        if iot["latest"] > 70:
            score += 10

    return round(min(score, 100), 1)

# ── MAIN ENGINE ─────────────────────────────────────────
def run():
    print("\n" + "="*60)
    print("  MARKET DEMAND INTELLIGENCE ENGINE")
    print(f"  GEO: {GEO}")
    print("="*60)

    tr = Trends()
    results = []

    for i, keyword in enumerate(SERVICE_KEYWORDS, 1):
        print(f"\n[{i}/{len(SERVICE_KEYWORDS)}] Analyzing: {keyword}")

        regions = fetch_regions(tr, keyword)
        related = fetch_related(tr, keyword)
        iot = fetch_iot(tr, keyword)

        opportunity_score = score_opportunity(
            regions, related["rising"], iot
        )

        entry = {
            "keyword": keyword,
            "regions": regions,
            "top_queries": related["top"],
            "rising_queries": related["rising"],
            "trend_data": iot,
            "opportunity_score": opportunity_score,
            "timestamp": datetime.datetime.utcnow().isoformat() + "Z"
        }

        results.append(entry)

    # Sort best opportunities first
    results.sort(key=lambda x: x["opportunity_score"], reverse=True)

    # ── PRINT RESULTS ───────────────────────────────────
    print("\n" + "="*60)
    print("  TOP MARKET OPPORTUNITIES")
    print("="*60)

    for i, r in enumerate(results, 1):
        print(f"\n#{i} {r['keyword']}")
        print(f"   Score      : {r['opportunity_score']}/100")
        print(f"   Regions    : {', '.join(d['region'] for d in r['regions'][:3])}")
        print(f"   Rising     : {', '.join(r['rising_queries'][:3])}")
        print(f"   Momentum   : {r['trend_data'].get('latest', 'N/A')}")

    # ── SAVE JSON ───────────────────────────────────────
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump({
            "meta": {
                "geo": GEO,
                "generated": datetime.datetime.utcnow().isoformat() + "Z",
                "count": len(results)
            },
            "data": results
        }, f, indent=2)

    print(f"\n✅ Report saved to {OUTPUT_FILE}\n")

    return results


# ── NOTEBOOK HELPER ─────────────────────────────────────
def run_notebook():
    results = run()
    rows = []

    for r in results:
        rows.append({
            "Keyword": r["keyword"],
            "Score": r["opportunity_score"],
            "Top Regions": " | ".join(d["region"] for d in r["regions"][:3]),
            "Rising Queries": " | ".join(r["rising_queries"][:3]),
            "Momentum": r["trend_data"].get("latest", "N/A"),
        })

    return pd.DataFrame(rows)


if __name__ == "__main__":
    run()